<a href="https://www.kaggle.com/code/najeebullah160/phase0-setup?scriptVersionId=372096909" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Phase 0 — Environment Setup & LLM Proof of Concept (Kaggle)  
**Project:** Explainable-NIDS — GraphIDS reproduction on NF-UNSW-NB15-v3  
**Platform:** Kaggle Notebooks, **CPU session** (GPU gate happens in Phase 2 — saves weekly quota)  
**Data:** private Kaggle dataset nf-unsw-nb15-v3 (official UQ download, CC BY-NC-SA 4.0)  
**LLM:** Groq API via Kaggle Secrets  
**Gate:** this notebook must run end-to-end before Phase 1 starts.

In [1]:
# === ENVIRONMENT CHECK (Kaggle CPU session) ===
import os, platform, shutil

print("Python    :", platform.python_version())
print("CPU cores :", os.cpu_count())
print("RAM       : ~%.0f GB" % (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9))
print("Disk free : %.1f GB in /kaggle/working" % (shutil.disk_usage('/kaggle/working').free / 1e9))
!nvidia-smi || echo "No GPU in this session — EXPECTED on CPU. The GPU gate moves to Phase 2."

Python    : 3.13.15
CPU cores : 4
RAM       : ~34 GB
Disk free : 20.9 GB in /kaggle/working
/bin/bash: line 1: nvidia-smi: command not found
No GPU in this session — EXPECTED on CPU. The GPU gate moves to Phase 2.


In [2]:
# === CLONE GraphIDS + INSTALL DEPENDENCIES ===
![ -d /kaggle/working/GraphIDS ] || git clone https://github.com/lorenzo9uerra/GraphIDS.git
%cd /kaggle/working/GraphIDS
!pip install -q uv
!uv sync

Cloning into 'GraphIDS'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 385 (delta 118), reused 96 (delta 86), pack-reused 234 (from 1)
Receiving objects: 100% (385/385), 20.95 MiB | 32.55 MiB/s, done.
Resolving deltas: 100% (226/226), done.
/kaggle/working/GraphIDS
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 52ms
Prepared 75 packages in 55.83s
░░░░░░░░░░░░░░░░░░░░ [0/75] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 75 packages in 22.58s
 + aiohappyeyeballs==2.6.1
 + aioht

In [3]:
# === STAGE DATASET: /kaggle/input is READ-ONLY, so copy into the layout GraphIDS expects ===
import glob, os, shutil

src_csv  = glob.glob('/kaggle/input/**/NF-UNSW-NB15-v3.csv', recursive=True)
src_feat = glob.glob('/kaggle/input/**/NetFlow_v3_Features.csv', recursive=True)
print("Found dataset CSV :", src_csv)
print("Found features CSV:", src_feat)
assert src_csv and src_feat, "Dataset not attached! Check Input panel -> Add Input, then re-run."

os.makedirs('data/NF-UNSW-NB15-v3', exist_ok=True)
for f in src_csv + src_feat:
    dst = os.path.join('data/NF-UNSW-NB15-v3', os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("Copied:", dst)

!ls -lh data/NF-UNSW-NB15-v3/

Found dataset CSV : ['/kaggle/input/datasets/najeebullah160/nf-unsw-nb15-v3/NF-UNSW-NB15-v3.csv']
Found features CSV: ['/kaggle/input/datasets/najeebullah160/nf-unsw-nb15-v3/NetFlow_v3_Features.csv']
Copied: data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv
Copied: data/NF-UNSW-NB15-v3/NetFlow_v3_Features.csv
total 551M
-rw-rw-rw- 1 root root 2.9K Nov 21 07:19 NetFlow_v3_Features.csv
-rw-rw-rw- 1 root root 551M Nov 21 07:19 NF-UNSW-NB15-v3.csv


In [4]:
# === VERIFY GraphIDS CLI ===
!uv run main.py --help

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
usage: main.py [-h] [--config CONFIG] --data_dir DATA_DIR
               [--checkpoint CHECKPOINT] [--seed SEED]
               [--dataset {NF-UNSW-NB15-v3,NF-CSE-CIC-IDS2018-v3,NF-UNSW-NB15-v2,NF-CSE-CIC-IDS2018-v2}]
               [--fanout FANOUT] [--edim_out EDIM_OUT]
               [--ae_embedding_dim AE_EMBEDDING_DIM]
               [--ae_feedforward_dim AE_FEEDFORWARD_DIM] [--dropout DROPOUT]
               [--ae_dropout AE_DROPOUT] [--learning_rate LEARNING_RATE]
               [--weight_decay WEIGHT_DECAY]
               [--ae_weight_decay AE_WEIGHT_DECAY] [--num_epochs NUM_EPOCHS]
               [--fraction FRACTION]
               [--split_mode {stratified,temporal,temporal_shift_aware}]
               [--distribution_segment {pre_shift,post_shift}]
               [--patience PATIENCE]
               [--positional_encoding {None,learnable,sinusoidal}]
               [--agg_t

In [5]:
# === DATASET SMOKE-TEST: first 1000 rows ===
import pandas as pd

df = pd.read_csv('data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv', nrows=1000)
print("Shape (first 1000 rows):", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape (first 1000 rows): (1000, 55)
Columns: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'S

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


In [6]:
# === LLM SETUP — Groq via Kaggle Secrets ===
!pip install -q groq
from kaggle_secrets import UserSecretsClient
from groq import Groq

client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
print("Groq client ready ✔")

Groq client ready ✔


In [7]:
# === CONNECTION TEST: list available Groq models ===
models = client.models.list()
for m in models.data:
    print(m.id)

meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
allam-2-7b
openai/gpt-oss-20b
openai/gpt-oss-120b
whisper-large-v3-turbo
whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi


In [8]:
# === LLM PoC #1 — can the LLM act as a security analyst? ===
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a network security analyst. Be concise."},
        {"role": "user", "content": "Explain what a NetFlow record is, in 2 sentences."}
    ],
    temperature=0.1,
)
print(response.choices[0].message.content)

A NetFlow record is a data structure that captures key attributes of a network flow—such as source/destination IPs, ports, protocol, and byte/packet counts—exported by routers or switches. It enables traffic analysis, capacity planning, and security monitoring by providing a summarized view of network activity.


In [9]:
# === LLM PoC #2 — analyze a realistic NetFlow record ===
from IPython.display import Markdown, display

flow = ("IPV4_SRC_ADDR: 192.168.1.5, IPV4_DST_ADDR: 10.0.0.7, L4_SRC_PORT: 443, "
        "L4_DST_PORT: 8080, PROTOCOL: 6, L7_PROTO: 7, IN_BYTES: 45236, OUT_BYTES: 812, "
        "IN_PKTS: 312, OUT_PKTS: 4, FLOW_DURATION_MILLISECONDS: 1200")

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a network security analyst. Analyze this NetFlow and explain in plain language whether it looks suspicious and why. Be honest about uncertainty."},
        {"role": "user", "content": flow}
    ],
    temperature=0.1,
)
display(Markdown(response.choices[0].message.content))

In [10]:
# === SAVE PoC ARTIFACT (persists as notebook output) ===
with open('/kaggle/working/poc_llm_explanation.md', 'w') as f:
    f.write(response.choices[0].message.content)
print("poc_llm_explanation.md saved ✔")

poc_llm_explanation.md saved ✔


# Phase 0 Gate — every box ✔ before Phase 1
[✔] Cell 1: Python, RAM, disk printed (no GPU — by design)  
[✔] Cell 2: clone + uv sync completed with no errors  
[✔] Cell 3: both CSVs found and staged (~550 MB visible in ls)  
[✔] Cell 4: main.py --help printed  
[✔] Cell 5: 1000 rows, ~53 columns  
[✔] Cell 8: model list printed  
[✔] Cells 9–10: LLM answered both prompts  
[✔] Cell 11: artifact saved  
[✔] Version saved (Save & Run All) + pushed to GitHub  
[✔] README updated with Kaggle link